In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)
    
# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /home/fre.gilad/source/AgentDaC/notebooks


---

## Env

In [2]:
from src.utils.experiment import prepare_environment

prepare_environment("../api_keys")

## Backend

In [3]:
import art

from openpipe.client import OpenPipe
from art.local import LocalBackend

op_client = OpenPipe()
print("OpenPipe client initialized")

backend = LocalBackend(in_process=True)
print("Local backend initialized")

OpenPipe client initialized
Local backend initialized


## Model

In [ ]:
from src.configs.art_model_config import configs as art_model_configs
from src.models import load_art_model, load_vllm_model

print("Available ART model configurations:")
print(art_model_configs.keys())

model_name = "unsloth/Qwen2.5-14B-Instruct"
# model_name = "unsloth/llama-3.1-8B-instruct"
project_name = "easy2hard_dac_v2"

Available ART model configurations:
dict_keys(['32B', 'Qwen/Qwen2.5-32B-Instruct', 'unsloth/Qwen2.5-32B-Instruct', 'unsloth/Qwen2.5-14B-Instruct', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit', 'unsloth/Qwen3-14B', 'unsloth/Qwen3-14B-bnb-4bit', 'unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit', 'unsloth/Llama-4-Scout-17B-16E-Instruct'])


In [7]:
from art.dev import InternalModelConfig, EngineArgs, InitArgs


model, dir_config = await load_art_model(
    model_name=model_name,
    project_name=project_name,
    backend=backend,
)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


ERROR:asyncio:Task was destroyed but it is pending!
task: <Task cancelling name='Task-12' coro=<Event.wait() running at /home/fre.gilad/.local/share/uv/python/cpython-3.11.12-linux-x86_64-gnu/lib/python3.11/asyncio/locks.py:213> wait_for=<Future cancelled>>


ERROR:asyncio:Task was destroyed but it is pending!
task: <Task cancelling name='Task-19' coro=<Event.wait() running at /home/fre.gilad/.local/share/uv/python/cpython-3.11.12-linux-x86_64-gnu/lib/python3.11/asyncio/locks.py:213> wait_for=<Future cancelled>>


==((====))==  Unsloth 2025.6.12: Fast Qwen2 patching. Transformers: 4.53.2. vLLM: 0.9.1.
   \\   /|    NVIDIA A40. Num GPUs = 1. Max memory: 47.334 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Your GPU cannot handle sequence lengths of 256 due to limited GPU memory.
Unsloth: Your GPU can only handle approximately the maximum sequence length of 256.
Unsloth: vLLM loading unsloth/Qwen2.5-14B-Instruct with actual GPU utilization = 28.65%
Unsloth: Your GPU has CUDA compute capability 8.6 with VRAM = 47.33 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 256. Num Sequences = 128.
Unsloth: vLLM's KV Cache can use up to 0.0 GB. Also swap space = 6 GB.
INFO 07-26 09:59:18 [config.py:823] This model 

RuntimeError: Sleep mode can only be used for one instance per process.

In [ ]:
# load_vllm_model(
#     model_name=model_name,
#     server_port=8200,
#     gpu_id=1,
# )

## Data

In [ ]:
from datasets import Dataset, load_dataset, DatasetDict

dataset_dict: DatasetDict = load_dataset(
    path="furonghuang-lab/Easy2Hard-Bench",
    name="E2H-AMC",
    split=None,
)  # type: ignore

train_data: Dataset = dataset_dict["train"]
test_data: Dataset = dataset_dict["eval"]

## Training Setup

### Inference Clients

In [ ]:
from src.vllm_client import VllmClient, VllmRouter, ArtVLLMClient

inference_clients: list[VllmClient] = [ArtVLLMClient(model)]

vllm_server_ports = []
for port in vllm_server_ports:
    vllm_client = VllmClient(port=port, base_model=model_name)
    inference_clients.append(vllm_client)

vllm_router: VllmRouter = VllmRouter(vllm_clients=inference_clients)

### Rewards Definition

In [ ]:
from src.utils.text import extract_text_between_markers
from src.configs.markers import Markers
from src.dac_agent import ChatMessage

def answer_reward(sample: dict[str, str], message: ChatMessage) -> float:
    role = message.role
    content = message.content

    if role != "assistant":
        raise ValueError(f"Expected role 'assistant', got '{role}'")

    prompt = sample["problem"]
    answer = sample["answer"]
    total_reward = 0.0

    answer_list = extract_text_between_markers(content, Markers.ANSWER_START, Markers.ANSWER_END)

    if len(answer_list) == 0:
        total_reward -= 3
        llm_answer = content.strip()

    elif len(answer_list) > 1:
        total_reward -= 0.5 * (len(answer_list) - 1)
        llm_answer = answer_list[-1].strip()

    else:
        llm_answer = answer_list[-1].strip()

    if llm_answer.lower().strip() == answer.lower().strip():
        total_reward += 1.5
    else:
        total_reward -= 1

    return total_reward


def format_reward(message: ChatMessage) -> float:
    
    role = message.role
    response = message.content

    if role != "assistant":
        raise ValueError(f"Expected role 'assistant', got '{role}'")

    reward = 0.0

    tasks = extract_text_between_markers(response, Markers.TASK_START, Markers.TASK_END)
    answers = extract_text_between_markers(response, Markers.ANSWER_START, Markers.ANSWER_END)

    if len(tasks) == 0 and len(answers) == 0:
        reward -= 0.1  # Penalize for no tasks or answers
    elif len(tasks) > 0 and len(answers) == 0:
        reward += 0.2  # ** len(tasks)  # Reward for tasks without answers
    elif len(tasks) == 0 and len(answers) > 0:
        reward += 0.2 ** len(answers)  # Reward for answers without tasks, diminishing with more answers
    else:
        reward -= 0.1 ** (
            1 / min(len(tasks), len(answers))
        )  # Penalize for each task that was also answered by the agent

    tasks_diff = abs(response.count(Markers.TASK_START) - response.count(Markers.TASK_END))
    answers_diff = abs(response.count(Markers.ANSWER_START) - response.count(Markers.ANSWER_END))

    if tasks_diff > 0:
        reward -= 0.1 ** (1 / tasks_diff)
    if answers_diff > 0:
        reward -= 0.1 ** (1 / answers_diff)

    return reward

### Trainer Implementation

In [ ]:
from src.dac_agent import AgentNode
from src.dac_agent_single import SingleAgentNode
from src.trainer import Trainer
from src.dac_agent import ChatMessage
from src.utils.text import extract_answer
from openai.types.chat.chat_completion import Choice


class Easy2HardTrainer(Trainer):
    def create_agent(self) -> AgentNode:
        client = self.get_client()
        return SingleAgentNode(
            model_name=client.get_inference_name(),
            client=client.client,
            prompt_config=self.prompt_config,
            stop_criteria=self.stop_criteria.clone(),
        )

    async def forward_step(self, sample: dict, **kwargs) -> art.Trajectory:
        agent = self.create_agent()
        message = ChatMessage(role="user", content=sample["problem"])
        trajectory = await agent.chat(message, **kwargs)
        return trajectory

    async def rollout_step(self, sample: dict) -> art.Trajectory:
        # Perform a forward step to get the trajectory
        trajectory = await self.forward_step(sample)
        ans_message = ChatMessage.model_validate(trajectory.messages()[-1], from_attributes=True)

        # Update rewards
        ans_reward = answer_reward(sample, ans_message)
        trajectory.reward += ans_reward

        for item in trajectory.messages_and_choices:
            if isinstance(item, Choice):
                msg = ChatMessage.model_validate(item.message, from_attributes=True)
                trajectory.reward += format_reward(msg)

        # Compute metadata and metrics
        problem = sample["problem"].strip()
        answer = sample["answer"].strip()
        agent_answer = extract_answer(ans_message.content)
        is_correct = 1 if answer == agent_answer.strip() else 0

        # Update metadata and metrics
        trajectory.metadata["problem"] = problem
        trajectory.metadata["answer"] = answer
        trajectory.metadata["agent_answer"] = agent_answer
        trajectory.metadata["item_difficulty"] = sample["item_difficulty"] 
        
        trajectory.metrics["is_correct"] = is_correct
        return trajectory

## Training and Inference 

In [ ]:
from src.trainer import TrainingConfig, PromptConfig, StopCriteria
from src.configs.prompts import DaCSystemPrompt


train_config = TrainingConfig(
    epochs=10,
    num_groups=2,
    group_size=10,
    config=art.types.TrainConfig(learning_rate=1e-5),
)

sys_prompt = PromptConfig(
    root_system_prompt=DaCSystemPrompt.dac_sys_prompt_gilad_root,
    inter_system_prompt=DaCSystemPrompt.dac_sys_prompt_gilad_inter,
    leaf_system_prompt=DaCSystemPrompt.dac_sys_prompt_gilad_leaf,
)

stop_criteria = StopCriteria(
    max_depth=1,
    max_tasks=5,
    max_rounds=5,
)

trainer = Easy2HardTrainer(
    model=model,
    client_list=inference_clients,
    dir_config=dir_config,
    train_config=train_config,
    prompt_config=sys_prompt,
    stop_criteria=stop_criteria,
)

### Inference Test

In [ ]:
import random

idx = random.randint(0, len(train_data) - 1)
# idx = 228
print(f"Selected random index: {idx}")
sample = train_data[idx]
problem = sample["problem"]
true_answer = sample["answer"]

print(f"Problem: {problem}")
print(f"Answer: {true_answer}")
print("-" * 200)
print()

preds = await trainer.predict([sample], verbose=True, seed=0)

### Train

In [ ]:
await trainer.train(
    train_dataset=train_data.to_list(),
    eval_dataset=test_data.to_list(),
)